In [3]:
import dataikuapi
from typing import Dict, Any
import os
import io
import fitz  # PyMuPDF
import pdfplumber
from uuid import uuid4
from soa_extraction.opensearch_utils import OpensearchUtil

import sys
import os
import fitz  # PyMuPDF
# from langgraph_utils.Info_extractor import InfoExtractorAgent
# from langgraph_utils.chat_history import SnowflakeChatMessageHistory
import json
# from langgraph_utils.file_parser import FileParser
# from langgraph_utils.digitization import main_handler
import uuid
# from langgraph_utils import creds
# from langgraph_utils.variables import PROJECT_NAME, SECRET_NAME, TOKEN_KEY
from utils.connection import get_dataiku_client_and_project
import logging



DATAIKU_HOST = "http://10.45.152.66:10000"
API_SECRET_KEY = "dkuaps-b3EsRXVjU3w4y7nd4KEwibEr04CjFPZr"          
PROJECT_NAME = "ECSGENERATION"   

# client, proj = get_dataiku_client_and_project(PROJECT_NAME, SECRET_NAME, TOKEN_KEY)
client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
proj = client.get_project(PROJECT_NAME)

In [4]:
def make_llm_call(form_name,field_name,client,proj):
    prompt = f"""
        You are a Edit check list specifcation specialist for Case Report Form .
        Given is an Example of how the data needs to be Generated this is a few shot example only
        `[
        vaildation_id : MVAL_DM027
        form_name : Demographics,
        'form_domain_name':DM
        'form_field_value':Country
        'variable_name': 'COUNTRY',
        'validation_logic': '(DM.COUNTRY is enterable and missing)', 'reasoning': 'Field must not be missing when enterable', 
        'action': 'prompt user with ACTION DETAILS',
        'action_details': '<query the field for missing data>',             
                ]`
        
        now you job is to create the json output for the these input fields :
        form name :{form_name} 
        field name :{field_name}
        
        Instruction:
        - Do not Expalin yourself , only json output
        - Do not change the data , generate for the form and field provided 
        - dont hallucinate 
        
    """
    prompt2 = f"""You are an Edit Check Specification Specialist for Case Report Forms (CRFs). You will receive a user input that contains two variables: form_name and field_name. Your ONLY job is to produce a JSON array of edit-check specification objects for the provided form_name and field_name. Follow these rules exactly:

1. OUTPUT FORMAT:
   - Return raw JSON only (no markdown, no code fences, no explanations, no extra text).
   - The top-level JSON must be an JSON. 
   - Generate Output for
   form name :{form_name} 
        field name :{field_name}

2. SCHEMA (each object must include exactly these keys):
   - validation_id
   - form_name
   - form_domain_name
   - form_field_value
   - variable_name
   - validation_logic
   - reasoning
   - action
   - action_details
   - source

   Do not add or remove keys.

4. DETERMINISTIC DERIVATIONS:
   - variable_name: derive by converting field_name to UPPERCASE snake_case (letters, numbers, underscores only). Example: "Date of Birth" -> "DATE_OF_BIRTH".
   - form_domain_name: map common forms (Demographics->DM, Medical History->MH, Vital Signs->VS, Adverse Event->AE, Concomitant Meds->CM, Informed Consent->IC, Physical Examination->PE, Laboratory->LB). If the form_name is not in the mapping, derive the domain by concatenating the first 2–3 letters of each significant word in the form_name and uppercasing (e.g., "Post-treatment Follow-up" -> "PTF").
   - validation_id: deterministic string "MVAL_{{form_name}}NNN" where NNN is a 3-digit sequence starting at 001. Use 001 unless other context is provided.

... (other rules unchanged) ...

7. NO HALLUCINATION:
   - Do not invent facts, values, mappings, or external knowledge not produced by the deterministic rules above.
   - If you cannot deterministically choose a validation_logic or domain from the input, do NOT invent — instead return this exact error structure (as the only element in the array):

     
       {{
         "error": "insufficient_input",
         "form_name": "{form_name}",
         "field_name": "{field_name}",
         "message": "Cannot generate validation logic deterministically for this field."
       }}
     

End of system instructions.
"""

    default_llm_model = proj.get_variables()['local'].get('default_llm_model')
        #self.default_llm_model = 'azureopenai:Azure-OpenAi:gpt-4o'
#         logging.info(f"[PlannerAgent] Initialized with LLM model: {self.default_llm_model}")
       
    llm = proj.get_llm(default_llm_model).as_langchain_llm(
        completion_settings={
        "temperature": 0,
                
                "timeout": 300,
            "max_tokens": 8192
            # 5 minutes
            }
        )
        
    output = llm.invoke(prompt2)
    return output

In [6]:
import os
import io
import fitz  # PyMuPDF
import pdfplumber
from uuid import uuid4


class historical_CRF:
    
    def __init__(self, client, proj, chunk_size=1000):
        self.proj = proj
        self.client = client
        self.s3_folder_dataset_id = proj.get_variables()['local'].get('file_upload') # change file upload 
        self.input_folder = proj.get_managed_folder(self.s3_folder_dataset_id)
        self.files = self.input_folder.list_contents()["items"]
        self.toc_page_limit = 20
        self.config = proj.get_variables()["local"]
        print(self.files)

    def historical_mapping(self, file_path,paths=[]):
        result = []

#         for file in paths:
#             path = file
#             print(path)
#             parts = path.strip('/').split('/')

#             if "Historical" in path:
                
#                 result.append({
                    
#                     "path": path,
                   
    
#                 })
        result.append(file_path)

        response = []

        for i in result:
            

            with self.input_folder.get_file(file_path) as stream:
                file_bytes = stream.raw.data
            
            
            try:
                
                import re

                def clean_summary(text):
                    # Remove newlines, tabs, and collapse extra spaces
                    text = re.sub(r'\s+', ' ', text).strip()

                    # Ensure it ends with a single period
                    if not text.endswith('.'):
                        text += '.'

                    return text
                
                pdf_file_like = io.BytesIO(file_bytes)
                with pdfplumber.open(pdf_file_like) as pdf:
                    
                    for page_num, page in enumerate(pdf.pages):
                        res = {}
#                         parts = i["path"].strip('/').split('/')
                        therapeutic_area = ''
                        source =  "Unknown"
                        template_name = os.path.basename(file_path)
                        unique_id = uuid4()
#                         print(hist_id)
                        res = {
                           
                            
                            "template_name": template_name,
                            "path": file_path,
                            "id": unique_id,

                        }
                        
                        page_text = page.extract_text(layout=True)
                        
                        if page_text and "field name" in page_text.lower():
                            continue

                        if not page_text:
                            continue

                        lines = page_text.split('\n')
                        header_lines = []
                        field_value_map = {}
                        current_field = ""
                        in_field_section = False

                        for line in lines:
                            line = line.strip()
                            if not line:
                                continue

                            # Trigger point for header vs fields
                            if not in_field_section:
                                if "generated" in line.lower():
                                    in_field_section = True
                                    continue
                                header_lines.append(line)
                            else:
                                if len(line.strip()) == 0:
                                    continue
                                
                                pattern = r'''
                                    ^                              # Start of line
                                    (?P<field>.+?)                 # Field name (non-greedy)
                                    (?:\t|\s{2,})+                 # Separator: tab or ≥2 spaces
                                    (?P<value>.+?)                 # Value
                                    \s*$                           # Optional trailing spaces
                                '''
                                pattern2  = r'^(?!\s)(?!.*\s$)(?P<value>.+)$'

                                
                                
                                
                                
                                
#                                 current_field = None

                                # Step 1 ─ collect every “proper” field line
                                m = re.match(pattern, line, re.VERBOSE)
                                p = re.match(pattern2, line, re.VERBOSE) 
                                if m:
                                    if m.group("field") and m.group("value"):
                                        feild = m.group("field")
                                        current_field = feild
                                        value = m.group("value")
                                        
                                        
                                if p:
                                    if p.group("value") and current_field:
                                        # Continuation of previous field
#                                         feild = current_field
                                        value = p.group("value")
                                        
                        
                               
                                
                                left_part = current_field.strip()
                                right_part = value.strip()

                                if left_part:
                                    if left_part in field_value_map and right_part:
                                        field_value_map[left_part].append(right_part)
                                     
                                    else:
                                        
                                        field_value_map[left_part] = [right_part.strip()]
                             

                        final_feilds = []
                        for k in field_value_map:
                            
                            final_feilds.append({
                                "field_name" : k,
                                "field_value": field_value_map[k]
                            })
                        
                        head = ""
                        for j in header_lines:
                            if "form" in j.lower().strip() or "folder" in j.lower().strip():
                                head += j + " "
                        
                        res["source_data"] = {
                            "assessments" : head,
                            "feilds" : final_feilds 
                            
                        }
#                         print(res)
                    
#                     print(res)
                        if final_feilds and head:
                            response.append(res)
#                             print(f"✅ extraction for {i['path']} completed")
                    

            except Exception as e:
                print(f"Error processing file {i['path']}: {e}")
        
        

        return response


In [7]:
obj = historical_CRF(client , proj)
file_path = '/Annotated_Otsuka_405 201 00157_00150405_v1.0_Complete eCRF (1).pdf'
import time
import re
start = time.time()
final_list = []
response = obj.historical_mapping(file_path)

for resp in response:
    form_name = resp['source_data']['assessments']
    match = re.search(r'Form[:\s]*(.*)', form_name)
    if match:
#         print(match.group(1))
        form_name = match.group(1)
    for field in resp['source_data']['feilds']:
        field_name = field['field_name']
        final_list.append({
            "form_name":form_name,
            "field_name":field_name
        })
end = time.time()
tt = end-start
import pandas as pd 
pd.set_option("display.max_rows",None)
data = pd.DataFrame(final_list)

[{'path': '/Annotated_Otsuka_405 201 00157_00150405_v1.0_Complete eCRF (1).pdf', 'size': 2544020, 'lastModified': 1757496080000}, {'path': '/output.xlsx', 'size': 671576, 'lastModified': 1757925327000}]


In [8]:
sub_data = data.iloc[:101]
print(tt)

38.89541220664978


In [31]:
# single search 
start = time.time()
opensearch_client = OpensearchUtil(client,proj)
client_os  = opensearch_client.opensearch_client
output_list = []
index_name = proj.get_variables()['local'].get('ecs_opensearch')
dataiku_project_var = "${projectKey}"
if dataiku_project_var in index_name:
    index_name = index_name.replace(dataiku_project_var, opensearch_client.project.project_key).lower()
for index , row in sub_data.iterrows():
    form_name = row['form_name']
    field_name = row['field_name']
    form_emb = opensearch_client.create_embedding(form_name,proj.get_variables()['local'].get("default_embeddings_model_id"))
    field_emb = opensearch_client.create_embedding(field_name,proj.get_variables()['local'].get("default_embeddings_model_id"))
    
    form_vec = form_emb['response']
    field_vec = field_emb['response']
    query = {
    "query": {
        "bool": {
            "should": [
                {
                    "knn": {
                        "form_name_vector": {
                            "vector": form_vec,
                            "k": 10
                        }
                    }
                },
                {
                    "knn": {
                        "form_field_value_vector": {
                            "vector": field_vec,
                            "k": 10
                        }
                    }
                }
            ]
        }
    }
}
    p = opensearch_client.opensearch_client
    result = p.search(index=index_name,body=query,size=1)
    for hit in result["hits"]["hits"]:
        print(hit["_id"], hit["_score"]/2)
        hit['_source']['original_form_name'] = form_name
        hit['_source']['original_field'] = field_name
        hit['_source']['score'] = hit['_score']/2
        print("-------------->>>>",hit['_score']/2)
        if hit['_score']/2 < 0.65:
            #make the llm call 
            output = json.loads(make_llm_call(form_name,field_name))
#             print(output)
            if isinstance(output,list):
                output = output[0]
                output['ecs_id'] = hit['_source']['ecs_id']
                output['form_id'] = hit['_source']['form_id']
                output['original_form_name'] = form_name
                output['original_field'] = field_name
                output['score'] = hit['_score']/2
                output['source'] = 'LLM Geneated'
            else:
                output['ecs_id'] = hit['_source']['ecs_id']
                output['form_id'] = hit['_source']['form_id']
                output['original_form_name'] = form_name
                output['original_field'] = field_name
                output['score'] = hit['_score']/2
                output['source'] = 'LLM Geneated'
                
            output_list.append(output)
        else:   
            output_list.append(hit['_source'])
        
final_df = pd.DataFrame(output_list)
end = time.time()

{'type': 'ElasticSearch', 'params': {'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'username': 'genai-admin', 'password': 'qPk7Jf5vcyXDS!**332gXTvSfmcauvr9', 'port': 443, 'ssl': True, 'trustAnySSLCertificate': True, 'dialect': 'ES_7', 'dkuProperties': [], 'namingRule': {'indexNameDatasetNamePrefix': '${projectKey}_'}, 'authType': 'PASSWORD', 'oauth': {'refreshTokenRotation': False}, 'aws': {'service': 'OPENSEARCH_SERVERLESS', 'credentialsMode': 'KEYPAIR', 'customAWSCredentialsProviderParams': []}}, 'credentialsMode': 'GLOBAL', 'proxySettingsAsString': ''}
opensearch <OpenSearch([{'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'port': 443}])>
44c40b95-e715-4892-bcd4-6400447a965e 0.5997956
-------------->>>> 0.5997956


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/opensearchpy/connection/http_urllib3.py:214: UserWarning: Connecting to https://aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com:443 using SSL with verify_certs=False is insecure.
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verific

8eeef31a-97fa-4fef-bcb7-a4b1c72b1369 0.6237687
-------------->>>> 0.6237687


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


8eeef31a-97fa-4fef-bcb7-a4b1c72b1369 0.6223632
-------------->>>> 0.6223632


...

dbffdc0c-e6ce-497d-9ead-64e42bf054b3 0.60414445
-------------->>>> 0.60414445


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


083c4b15-6292-4282-85f0-323d2349e15b 0.61819185
-------------->>>> 0.61819185


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


9e1e2d75-eec2-4ce6-90fa-ffc6f09bb39d 0.58385105
-------------->>>> 0.58385105


In [14]:
#llm out put 
import json
field = 'Did participant satisfy all Inclusion/Exclusion criteria?'
form = 'Inclusion/Exclusion Criteria	'
out = make_llm_call(form,field)

In [32]:
end - start
len(final_df)

101

In [33]:
# final_df.to_excel()
import pandas as pd
import io

# Sample DataFrame
df = final_df

# Convert to Excel bytes
buffer = io.BytesIO()
with pd.ExcelWriter(buffer, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="Sheet1")

excel_bytes = buffer.getvalue()  # <-- This is your Excel file as bytes

obj.input_folder.put_file("/output.xlsx",excel_bytes)

# Example: write to disk (optional)
with open("output.xlsx", "wb") as f:
    f.write(excel_bytes)


In [28]:
res_df = final_df[["original_form_name", "original_field","form_name","form_field_value","score"]]
res_dfres_df = final_df[["original_form_name", "original_field","form_name","form_field_value","score"]]
res_df

,original_form_name,original_field,form_name,form_field_value,score
0,Enrollment,Site ID,Enrollment,Site ID,0.599796
1,Enrollment,Site ID,Demographics,Country,0.599796
2,Enrollment,Participant ID,Enrollment,Participant ID,0.623769
3,Enrollment,Participant ID,Demographics,Age Unit,0.623769
4,Enrollment,Participant Number (Derived),Enrollment,Participant Number (Derived),0.622363
5,Enrollment,Participant Number (Derived),Demographics,Age Unit,0.622363
6,Date of Visit,Visit date,Subject Visits,Visit Date,0.792303
7,Informed Consent,Informed consent obtained?,Informed Consent,Was informed consent obtained?,0.923950
8,Informed Consent,Informed consent date,Informed Consent,Was informed consent obtained?,0.846530
9,Informed Consent,Informed consent time,Informed Consent,Was informed consent obtained?,0.833227


In [21]:
client

<OpenSearch([{'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'port': 443}])>

In [29]:
len(sub_data)

101

In [30]:
final_df.drop_duplicates(subset=['form_name','form_field_value'])

,validation_id,form_name,form_domain_name,form_field_value,variable_name,validation_logic,reasoning,action,action_details,source,...,form_id,original_form_name,original_field,score,path,form_name_vector,form_field_value_vector,error,field_name,message
0,MVAL_EnrollmentNNN001,Enrollment,EN,Site ID,SITE_ID,NOT MISSING,Site ID is a critical identifier and should al...,Query,Please provide the Site ID,LLM Geneated,...,4a5552de-4909-4846-bd25-3c28f9351b53,Enrollment,Site ID,0.599796,NaN,NaN,NaN,NaN,NaN,NaN
1,MVAL_DM027,Demographics,DM,Country,COUNTRY,(DM.COUNTRY is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,...,4a5552de-4909-4846-bd25-3c28f9351b53,Enrollment,Site ID,0.599796,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.00561428, 0.014669912, 0.0441228, 0.056164,...","[-0.01750083, -0.014178229, 0.03174632, 0.0935...",NaN,NaN,NaN
2,MVAL_EnrollmentNNN001,Enrollment,EN,Participant ID,PARTICIPANT_ID,NOT MISSING AND UNIQUE,Participant ID is a critical identifier and mu...,Query,Please provide a unique Participant ID.,LLM Geneated,...,4a5552de-4909-4846-bd25-3c28f9351b53,Enrollment,Participant ID,0.623769,NaN,NaN,NaN,NaN,NaN,NaN
3,MVAL_DM018,Demographics,DM,Age Unit,AGEU,(DM.AGEU is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,...,4a5552de-4909-4846-bd25-3c28f9351b53,Enrollment,Participant ID,0.623769,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.00561428, 0.014669912, 0.0441228, 0.056164,...","[-0.022254938, 0.017848019, 0.031054085, 0.029...",NaN,NaN,NaN
4,MVAL_EnrollmentNNN001,Enrollment,EN,Participant Number (Derived),PARTICIPANT_NUMBER_DERIVED,NOT NULL AND MATCHES_PATTERN('^[0-9]{3}-[0-9]{...,Participant Number is a critical identifier an...,Query,Please verify the Participant Number format. I...,LLM Geneated,...,4a5552de-4909-4846-bd25-3c28f9351b53,Enrollment,Participant Number (Derived),0.622363,NaN,NaN,NaN,NaN,NaN,NaN
6,MVAL_SV010,Subject Visits,SV,Visit Date,VISDAT,(SV.VISDAT is an invalid date),Field must not be an invalid date such as 31Fe...,prompt user with ACTION DETAILS,<query the field for invalid date>,Standard,...,3c36705a-c794-48a0-aa54-0453ea51bde5,Date of Visit,Visit date,0.792303,/Standard/Copy of Otsuka Standard Edit Check S...,"[-0.027394814416766167, 0.018084557726979256, ...","[-0.020077953, 0.0069108373, -0.023404991, -0....",NaN,NaN,NaN
7,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,...,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent obtained?,0.923950,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",NaN,NaN,NaN
10,MVAL_MH023,Medical History,MH,End Date,MHENDAT,(MH.MHENDAT is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,...,bd0860d8-e7ce-4589-b23b-cbcd5931650b,Informed Consent,Derived date,0.651459,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.021205350756645203, -0.014403634704649448, ...","[0.023139598, 0.020193553, 0.06760691, -0.0191...",NaN,NaN,NaN
12,MVAL_DS_IC004,Informed Consent,DS_IC,Type of Consent,DSSCAT,(DS_IC.DSSCAT is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,...,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Standardized disposition term,0.760939,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.028999867, 0.044233263, 0.0076308018, 0.063...",NaN,NaN,NaN
13,MVAL_DS_IC005,Informed Consent,DS_IC,Protocol Version Number,Q

In [35]:
# Replace 'your_column_name' with the actual column name you want to filter
filtered_df = final_df[final_df['source'] == 'Standard']


In [36]:
filtered_df

,validation_id,form_name,form_domain_name,form_field_value,variable_name,validation_logic,reasoning,action,action_details,source,...,form_id,original_form_name,original_field,score,path,form_name_vector,form_field_value_vector,error,field_name,message
3,MVAL_SV010,Subject Visits,SV,Visit Date,VISDAT,(SV.VISDAT is an invalid date),Field must not be an invalid date such as 31Fe...,prompt user with ACTION DETAILS,<query the field for invalid date>,Standard,...,3c36705a-c794-48a0-aa54-0453ea51bde5,Date of Visit,Visit date,0.792303,/Standard/Copy of Otsuka Standard Edit Check S...,"[-0.027394814416766167, 0.018084557726979256, ...","[-0.020077953, 0.0069108373, -0.023404991, -0....",NaN,NaN,NaN
4,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,...,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent obtained?,0.923950,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",NaN,NaN,NaN
5,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,...,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent date,0.846530,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",NaN,NaN,NaN
6,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,...,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent time,0.833227,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",NaN,NaN,NaN
7,MVAL_MH023,Medical History,MH,End Date,MHENDAT,(MH.MHENDAT is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,...,bd0860d8-e7ce-4589-b23b-cbcd5931650b,Informed Consent,Derived date,0.651459,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.021205350756645203, -0.014403634704649448, ...","[0.023139598, 0.020193553, 0.06760691, -0.0191...",NaN,NaN,NaN
8,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,...,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent version number,0.830627,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",NaN,NaN,NaN
9,MVAL_DS_IC004,Informed Consent,DS_IC,Type of Consent,DSSCAT,(DS_IC.DSSCAT is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,...,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Standardized disposition term,0.760939,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.028999867, 0.044233263, 0.0076308018, 0.063...",NaN,NaN,NaN
10,MVAL_DS_IC005,Informed Consent,DS_IC,Protocol Version Number,QVAL_PROTVER,(DS_IC.QVAL_PROTVER is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,...,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Protocol version,0.905990,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.0226

In [37]:
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz
test = "Informed consent date" # CRF
test2 = "Date of Consent"
# test2 = "Was informed consent obtained?"
out1 = opensearch_client.create_embedding(test,proj.get_variables()['local'].get("default_embeddings_model_id"))
out2 = opensearch_client.create_embedding(test2,proj.get_variables()['local'].get("default_embeddings_model_id"))
text_score = fuzz.token_sort_ratio(test, test2) / 100
print(text_score)
vec1 = [out1['response']]
vec2 = [out2['response']]
# print(vec2)
cos_sim = cosine_similarity(vec1, vec2)[0][0]
final_score = 0.7 * cos_sim + 0.3 * text_score
print("Cosine Similarity:", final_score)
#  0.656939349035508
# 0.6409953083535772

0.5555555555555556
Cosine Similarity: 0.656939349035508


In [14]:
final_df2.iloc[:10]

,validation_id,form_name,form_domain_name,form_field_value,variable_name,validation_logic,reasoning,action,action_details,source,ecs_id,form_id,original_form_name,original_field,score,path,form_name_vector,form_field_value_vector
0,MVAL_EnrollmentNNN001,Enrollment,EN,Site ID,SITE_ID,SITE_ID must be a valid site identifier,Ensure the entered Site ID is valid and corres...,Query,Please verify the entered Site ID is correct a...,LLM Generated,5322d11a-79e3-49a7-838c-0469661a9d4d,3c36705a-c794-48a0-aa54-0453ea51bde5,Enrollment,Site ID,0.312996,NaN,NaN,NaN
1,MVAL_EnrollmentNNN001,Enrollment,EN,Participant ID,PARTICIPANT_ID,NOT MISSING AND UNIQUE,Participant ID is a critical identifier and mu...,Query,Please provide a unique Participant ID,LLM Generated,8eeef31a-97fa-4fef-bcb7-a4b1c72b1369,4a5552de-4909-4846-bd25-3c28f9351b53,Enrollment,Participant ID,0.623769,NaN,NaN,NaN
2,MVAL_EnrollmentNNN001,Enrollment,EN,Participant Number (Derived),PARTICIPANT_NUMBER_DERIVED,NOT NULL AND UNIQUE,Participant Number is a critical identifier an...,Query,Please provide a unique Participant Number.,LLM Generated,8eeef31a-97fa-4fef-bcb7-a4b1c72b1369,4a5552de-4909-4846-bd25-3c28f9351b53,Enrollment,Participant Number (Derived),0.622363,NaN,NaN,NaN
3,MVAL_SV010,Subject Visits,SV,Visit Date,VISDAT,(SV.VISDAT is an invalid date),Field must not be an invalid date such as 31Fe...,prompt user with ACTION DETAILS,<query the field for invalid date>,Standard,5322d11a-79e3-49a7-838c-0469661a9d4d,3c36705a-c794-48a0-aa54-0453ea51bde5,Date of Visit,Visit date,0.792303,/Standard/Copy of Otsuka Standard Edit Check S...,"[-0.027394814416766167, 0.018084557726979256, ...","[-0.020077953, 0.0069108373, -0.023404991, -0...."
4,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent obtained?,0.923950,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630..."
5,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent date,0.846530,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630..."
6,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent time,0.833227,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630..."
7,MVAL_MH023,Medical History,MH,End Date,MHENDAT,(MH.MHENDAT is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,62ea8058-50d8-4eb0-9b1b-275ae89847b3,bd0860d8-e7ce-4589-b23b-cbcd5931650b,Informed Consent,Derived date,0.651459,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.021205350756645203, -0.014403634704649448, ...","[0.023139598, 0.020193553, 0.06760691, -0.0191..."
8,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,fda81295-a0ab-4279-8947-ac699d5ae1

In [29]:
# single search 
import time
from sklearn.metrics.pairwise import cosine_similarity
start = time.time()
opensearch_client = OpensearchUtil(client,proj)
client_os  = opensearch_client.opensearch_client
output_list = []
index_name = proj.get_variables()['local'].get('ecs_opensearch')
dataiku_project_var = "${projectKey}"
if dataiku_project_var in index_name:
    index_name = index_name.replace(dataiku_project_var, opensearch_client.project.project_key).lower()
for index , row in sub_data.iloc[:10].iterrows():
    form_name = row['form_name']
    field_name = row['field_name']
    form_emb = opensearch_client.create_embedding(form_name,proj.get_variables()['local'].get("default_embeddings_model_id"))
    field_emb = opensearch_client.create_embedding(field_name,proj.get_variables()['local'].get("default_embeddings_model_id"))
    
    form_vec = form_emb['response']
    field_vec = field_emb['response']
    query = {
    "query": {
        "bool": {
            "should": [
                {
                    "knn": {
                        "form_name_vector": {
                            "vector": form_vec,
                            "k": 10
                        }
                    }
                },
                {
                    "knn": {
                        "form_field_value_vector": {
                            "vector": field_vec,
                            "k": 10
                        }
                    }
                }
            ]
        }
    }
}
    p = opensearch_client.opensearch_client
    result = p.search(index=index_name,body=query,size=5)
    max_similarity = -1
    field_name_val_ = ''
    final_value = None
    for hit in result["hits"]["hits"]:
#         if hit["_source"]['form_name'].lower().strip() == "informed consent":
#             print(hit["_source"]['form_name'],hit['_source']['form_field_value'])
        cos_sim = cosine_similarity([field_vec], [hit['_source']['form_field_value_vector']])
        print("============",cos_sim,field_name,form_name,hit['_source']['form_field_value'])
        if cos_sim[0][0] > max_similarity:
            max_similarity = cos_sim[0][0]
#             if hit["_source"]['form_name'].lower().strip() == "informed consent":
#                 print("value",hit["_source"]['form_name'],hit['_source']['form_field_value'])
#                 print("score",cos_sim[0][0],hit["_source"]['form_name'],hit['_source']['form_field_value'])
            field_name_val_ = hit['_source']['form_field_value']
#             print('45678',field_name_val_,max_similarity)
            final_value = hit
            
    if final_value:  # ensure we got at least one hit
        final_value['_source']['original_form_name'] = form_name
        final_value['_source']['original_field'] = field_name
        final_value['_source']['score'] = final_value['_score']/2
#         print("-------------->>>>", final_value['_score']/2)

    if final_value['_score']/2 < 0.64:
        # make the llm call only once
        print('llm call')
        output = json.loads(make_llm_call(form_name, field_name))
        if isinstance(output, list):
            output = output[0]
        output['ecs_id'] = final_value['_source']['ecs_id']
        output['form_id'] = final_value['_source']['form_id']
        output['original_form_name'] = form_name
        output['original_field'] = field_name
        output['score'] = final_value['_score']/2
        output['source'] = 'LLM Generated'
        output_list.append(output)
    else:
        # use the matched value
        final_value['_source']['form_field_value'] = field_name_val_
        output_list.append(final_value['_source'])

        
    
    
        
final_df2 = pd.DataFrame(output_list)
end = time.time()

{'type': 'ElasticSearch', 'params': {'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'username': 'genai-admin', 'password': 'qPk7Jf5vcyXDS!**332gXTvSfmcauvr9', 'port': 443, 'ssl': True, 'trustAnySSLCertificate': True, 'dialect': 'ES_7', 'dkuProperties': [], 'namingRule': {'indexNameDatasetNamePrefix': '${projectKey}_'}, 'authType': 'PASSWORD', 'oauth': {'refreshTokenRotation': False}, 'aws': {'service': 'OPENSEARCH_SERVERLESS', 'credentialsMode': 'KEYPAIR', 'customAWSCredentialsProviderParams': []}}, 'credentialsMode': 'GLOBAL', 'proxySettingsAsString': ''}
opensearch <OpenSearch([{'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'port': 443}])>
============ [[0.2893107]] Site ID Enrollment  Country
============ [[0.32514917]] Site ID Enrollment  Severity
============ [[0.27207636]] Site ID Enrollment  Was the visit performed?
============ [[0.40253793]] Site ID Enrollment  Visit Date
============ [[0.40253793]] Site ID

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/opensearchpy/connection/http_urllib3.py:214: UserWarning: Connecting to https://aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com:443 using SSL with verify_certs=False is insecure.
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verific

============ [[0.41898702]] Participant ID Enrollment  Age Unit
============ [[0.39157055]] Participant ID Enrollment  Collection Date
============ [[0.39157055]] Participant ID Enrollment  Collection Date
============ [[0.39157055]] Participant ID Enrollment  Collection Date
============ [[0.35387279]] Participant ID Enrollment  Signed by
llm call


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


============ [[0.41192954]] Participant Number (Derived) Enrollment  Age Unit
============ [[0.36898755]] Participant Number (Derived) Enrollment  If Other, specify race
============ [[0.36898755]] Participant Number (Derived) Enrollment  If Other, specify race
============ [[0.33043366]] Participant Number (Derived) Enrollment  Collection Date
============ [[0.33043366]] Participant Number (Derived) Enrollment  Collection Date
llm call


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made t

============ [[0.88064105]] Visit date Date of Visit  Visit Date
============ [[0.88064105]] Visit date Date of Visit  Visit Date
============ [[0.88064105]] Visit date Date of Visit  Visit Date
============ [[0.88064105]] Visit date Date of Visit  Visit Date
============ [[0.88064105]] Visit date Date of Visit  Visit Date
============ [[0.92756998]] Informed consent obtained? Informed Consent  Was informed consent obtained?
============ [[0.49470924]] Informed consent obtained? Informed Consent  Type of Consent
============ [[0.49470924]] Informed consent obtained? Informed Consent  Type of Consent
============ [[0.49297655]] Informed consent obtained? Informed Consent  Date of Consent
============ [[0.49297655]] Informed consent obtained? Informed Consent  Date of Consent
============ [[0.71402691]] Informed consent date Informed Consent  Was informed consent obtained?
============ [[0.70038955]] Informed consent date Informed Consent  Date of Consent
============ [[0.70038955]] Info

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made t

============ [[0.66846957]] Informed consent time Informed Consent  Was informed consent obtained?
============ [[0.63357583]] Informed consent time Informed Consent  Time of Consent
============ [[0.63357583]] Informed consent time Informed Consent  Time of Consent
============ [[0.56356252]] Informed consent time Informed Consent  Date of Consent
============ [[0.56356252]] Informed consent time Informed Consent  Date of Consent
============ [[0.52019689]] Derived date Informed Consent  End Date
============ [[0.52019689]] Derived date Informed Consent  End Date
============ [[0.52019689]] Derived date Informed Consent  End Date
============ [[0.49967118]] Derived date Informed Consent  Collection Date
============ [[0.48451223]] Derived date Informed Consent  Collection Date (DD-MON-YYYY)
============ [[0.65918489]] Informed consent version number Informed Consent  Was informed consent obtained?
============ [[0.52610641]] Informed consent version number Informed Consent  Protocol V

In [28]:
final_df2

,validation_id,form_name,form_domain_name,form_field_value,variable_name,validation_logic,reasoning,action,action_details,source,ecs_id,form_id,original_form_name,original_field,score,path,form_name_vector,form_field_value_vector
0,MVAL_EnrollmentNNN001,Enrollment,EN,Site ID,SITE_ID,SITE_ID must be a valid site identifier,Ensure that the entered Site ID is valid and c...,Query,Please verify the entered Site ID. If incorrec...,LLM Generated,5322d11a-79e3-49a7-838c-0469661a9d4d,3c36705a-c794-48a0-aa54-0453ea51bde5,Enrollment,Site ID,0.312996,NaN,NaN,NaN
1,MVAL_Enrollment001,Enrollment,EN,Participant ID,PARTICIPANT_ID,NOT MISSING,Participant ID is a critical identifier and sh...,Query,Please provide the Participant ID,LLM Generated,8eeef31a-97fa-4fef-bcb7-a4b1c72b1369,4a5552de-4909-4846-bd25-3c28f9351b53,Enrollment,Participant ID,0.623769,NaN,NaN,NaN
2,MVAL_EnrollmentNNN001,Enrollment,EN,Participant Number (Derived),PARTICIPANT_NUMBER_DERIVED,NOT NULL AND MATCHES_PATTERN('^[0-9]{3}-[0-9]{...,Participant numbers are typically required and...,Query,Please verify the participant number format. I...,LLM Generated,8eeef31a-97fa-4fef-bcb7-a4b1c72b1369,4a5552de-4909-4846-bd25-3c28f9351b53,Enrollment,Participant Number (Derived),0.622363,NaN,NaN,NaN
3,MVAL_SV010,Subject Visits,SV,Visit Date,VISDAT,(SV.VISDAT is an invalid date),Field must not be an invalid date such as 31Fe...,prompt user with ACTION DETAILS,<query the field for invalid date>,Standard,5322d11a-79e3-49a7-838c-0469661a9d4d,3c36705a-c794-48a0-aa54-0453ea51bde5,Date of Visit,Visit date,0.792303,/Standard/Copy of Otsuka Standard Edit Check S...,"[-0.027394814416766167, 0.018084557726979256, ...","[-0.020077953, 0.0069108373, -0.023404991, -0...."
4,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent obtained?,0.923950,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630..."
5,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent date,0.846530,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630..."
6,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent time,0.833227,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630..."
7,MVAL_MH023,Medical History,MH,End Date,MHENDAT,(MH.MHENDAT is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,62ea8058-50d8-4eb0-9b1b-275ae89847b3,bd0860d8-e7ce-4589-b23b-cbcd5931650b,Informed Consent,Derived date,0.651459,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.021205350756645203, -0.014403634704649448, ...","[0.023139598, 0.020193553, 0.06760691, -0.0191..."
8,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,fda81295-a0ab-427

In [60]:
td = final_df2[["ecs_id","form_id","validation_id","form_name","form_domain_name","form_field_value","variable_name","validation_logic","reasoning","action","action_details"]]

In [78]:
filtered_df = final_df2[final_df2['source'] == "Standard"]

filtered_df

,validation_id,form_name,form_domain_name,form_field_value,variable_name,validation_logic,reasoning,action,action_details,source,...,form_id,original_form_name,original_field,score,path,form_name_vector,form_field_value_vector,error,field_name,message
3,MVAL_SV010,Subject Visits,SV,Visit Date,VISDAT,(SV.VISDAT is an invalid date),Field must not be an invalid date such as 31Fe...,prompt user with ACTION DETAILS,<query the field for invalid date>,Standard,...,3c36705a-c794-48a0-aa54-0453ea51bde5,Date of Visit,Visit date,0.792303,/Standard/Copy of Otsuka Standard Edit Check S...,"[-0.027394814416766167, 0.018084557726979256, ...","[-0.020077953, 0.0069108373, -0.023404991, -0....",NaN,NaN,NaN
4,MVAL_SV010,Subject Visits,SV,Visit Date,VISDAT,(SV.VISDAT is an invalid date),Field must not be an invalid date such as 31Fe...,prompt user with ACTION DETAILS,<query the field for invalid date>,Standard,...,3c36705a-c794-48a0-aa54-0453ea51bde5,Date of Visit,Visit date,0.792303,/Standard/Copy of Otsuka Standard Edit Check S...,"[-0.027394814416766167, 0.018084557726979256, ...","[-0.020077953, 0.0069108373, -0.023404991, -0....",NaN,NaN,NaN
5,MVAL_SV010,Subject Visits,SV,Visit Date,VISDAT,(SV.VISDAT is an invalid date),Field must not be an invalid date such as 31Fe...,prompt user with ACTION DETAILS,<query the field for invalid date>,Standard,...,3c36705a-c794-48a0-aa54-0453ea51bde5,Date of Visit,Visit date,0.792303,/Standard/Copy of Otsuka Standard Edit Check S...,"[-0.027394814416766167, 0.018084557726979256, ...","[-0.020077953, 0.0069108373, -0.023404991, -0....",NaN,NaN,NaN
6,MVAL_SV010,Subject Visits,SV,Visit Date,VISDAT,(SV.VISDAT is an invalid date),Field must not be an invalid date such as 31Fe...,prompt user with ACTION DETAILS,<query the field for invalid date>,Standard,...,3c36705a-c794-48a0-aa54-0453ea51bde5,Date of Visit,Visit date,0.792303,/Standard/Copy of Otsuka Standard Edit Check S...,"[-0.027394814416766167, 0.018084557726979256, ...","[-0.020077953, 0.0069108373, -0.023404991, -0....",NaN,NaN,NaN
7,MVAL_SV010,Subject Visits,SV,Visit Date,VISDAT,(SV.VISDAT is an invalid date),Field must not be an invalid date such as 31Fe...,prompt user with ACTION DETAILS,<query the field for invalid date>,Standard,...,3c36705a-c794-48a0-aa54-0453ea51bde5,Date of Visit,Visit date,0.792303,/Standard/Copy of Otsuka Standard Edit Check S...,"[-0.027394814416766167, 0.018084557726979256, ...","[-0.020077953, 0.0069108373, -0.023404991, -0....",NaN,NaN,NaN
8,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,...,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent obtained?,0.923950,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",NaN,NaN,NaN
9,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,...,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent obtained?,0.923950,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",NaN,NaN,NaN
10,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,...,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent obtained?,0.923950,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",NaN,NaN,NaN
11,MVAL_DS_IC006,I

In [79]:
# for hit in result["hits"]["hits"]:
#         print(hit["_id"], hit["_score"]/2,hit['_source']['form_name'],)
#         hit['_source']['original_form_name'] = form_name
#         hit['_source']['original_field'] = field_name
#         hit['_source']['score'] = hit['_score']/2
#         print("-------------->>>>",hit['_score']/2)
#         if final_value['_score']/2 < 0.64:
#             #make the llm call 
#             print('llm call')
#             output = json.loads(make_llm_call(form_name,field_name))
# #             print(output)
#             if isinstance(output,list):
#                 output = output[0]
#                 output['ecs_id'] = final_value['_source']['ecs_id']
#                 output['form_id'] = final_value['_source']['form_id']
#                 output['original_form_name'] = form_name
#                 output['original_field'] = field_name
#                 output['score'] = final_value['_score']/2
#                 output['source'] = 'LLM Geneated'
#             else:
#                 output['ecs_id'] = final_value['_source']['ecs_id']
#                 output['form_id'] = final_value['_source']['form_id']
#                 output['original_form_name'] = form_name
#                 output['original_field'] = field_name
#                 output['score'] = final_value['_score']/2
#                 output['source'] = 'LLM Geneated'
                
#             output_list.append(output)
#             break
#         else:   
# #             hit['_source']['form_field_value'] = field_name_val_
#             final_value['_source']['form_field_value'] = field_name_val_
#             output_list.append(final_value['_source'])

249